# Punto 5 — Preparación de datos para Machine Learning

**Continuación del caso base normalizado.**  
Objetivo: definir variable respuesta (VIOLENCIA_FISICA), variables predictoras, dividir en conjunto de entrenamiento y prueba con estratificación, y analizar el balance de clases para el informe.

In [8]:
import pandas as pd
from pathlib import Path
import pyreadstat
from sklearn.model_selection import train_test_split

## 1. Carga y preparación de datos (violencia intrafamiliar)

Se replica la misma carga y mapeo de departamentos que en el notebook **caso_base_normalizado**: archivos `.sav` de violencia, columna `ANIO` y columna `Departamento` desde `DEPTO_MCPIO`.

In [9]:
DEPARTAMENTOS_MAP = {
    1: 'Guatemala', 2: 'El Progreso', 3: 'Sacatepéquez', 4: 'Chimaltenango',
    5: 'Escuintla', 6: 'Santa Rosa', 7: 'Sololá', 8: 'Totonicapán',
    9: 'Quetzaltenango', 10: 'Suchitepéquez', 11: 'Retalhuleu', 12: 'San Marcos',
    13: 'Huehuetenango', 14: 'Quiché', 15: 'Baja Verapaz', 16: 'Alta Verapaz',
    17: 'Petén', 18: 'Izabal', 19: 'Zacapa', 20: 'Chiquimula',
    21: 'Jalapa', 22: 'Jutiapa'
}


def cargar_violencia(ruta_carpeta='data/violencia-intrafamiliar'):
    archivos = sorted(Path(ruta_carpeta).glob('*.sav'))
    listas = []
    for archivo in archivos:
        df_t, _ = pyreadstat.read_sav(str(archivo))
        anio = int(archivo.stem.split('-')[0])
        df_t['ANIO'] = anio
        listas.append(df_t)
    df = pd.concat(listas, ignore_index=True)
    codigo_depto = df['DEPTO_MCPIO'].dropna().astype(int).floordiv(100)
    df['Departamento'] = codigo_depto.map(DEPARTAMENTOS_MAP).reindex(df.index)
    return df


df_violencia = cargar_violencia()
print(f"Registros cargados: {len(df_violencia):,}")
print(f"Periodo: {df_violencia['ANIO'].min()}-{df_violencia['ANIO'].max()}")
print(f"Departamento mapeado: {df_violencia['Departamento'].notna().sum():,} ({df_violencia['Departamento'].notna().mean()*100:.1f}%)")

Registros cargados: 387,257
Periodo: 2009-2020
Departamento mapeado: 387,257 (100.0%)


## 2. Variable respuesta: VIOLENCIA_FISICA

Se deriva de **HEC_TIPAGRE** (tipo de agresión): los códigos cuyo primer dígito es 1 incluyen agresión física (1222, 1122, 1212, 1111, etc.). Codificación numérica estándar para clasificación: **1** = violencia física, **0** = no violencia física. Se excluyen filas con valor faltante o no válido en HEC_TIPAGRE.

In [10]:
def construir_target_violencia_fisica(dataframe):
    tipagre = pd.to_numeric(dataframe['HEC_TIPAGRE'], errors='coerce')
    mascara_valida = tipagre.notna() & (tipagre >= 1000) & (tipagre < 10000)
    serie = pd.Series(index=dataframe.index, dtype=object)
    serie.loc[mascara_valida] = (
        (tipagre.loc[mascara_valida].astype(int) // 1000) == 1
    ).map({True: 1, False: 0})
    return serie


df_violencia['VIOLENCIA_FISICA'] = construir_target_violencia_fisica(df_violencia)
filas_validas = df_violencia['VIOLENCIA_FISICA'].notna()
df_ml = df_violencia.loc[filas_validas].copy()
print("Variable respuesta VIOLENCIA_FISICA (1 = violencia física, 0 = no):")
print(df_ml['VIOLENCIA_FISICA'].value_counts().sort_index())
print(f"\nRegistros con target válido: {len(df_ml):,}")

Variable respuesta VIOLENCIA_FISICA (1 = violencia física, 0 = no):
VIOLENCIA_FISICA
0    154306
1    232951
Name: count, dtype: int64

Registros con target válido: 387,257


## 3. Variables predictoras (features) y matriz de trabajo

Se seleccionan columnas sociodemográficas, geográficas y temporales. Se eliminan filas con NaN en alguna feature o en el target para obtener el dataset final para el split.

In [11]:
COLUMNAS_FEATURES = [
    'VIC_SEXO', 'VIC_EDAD', 'VIC_ALFAB', 'VIC_ESCOLARIDAD',
    'VIC_EST_CIV', 'VIC_GRUPET', 'Departamento', 'ANIO'
]


def construir_matriz_features(dataframe, columnas):
    disponibles = [c for c in columnas if c in dataframe.columns]
    return dataframe[disponibles].copy()


features = construir_matriz_features(df_ml, COLUMNAS_FEATURES)
target = df_ml['VIOLENCIA_FISICA']
columnas_usadas = list(features.columns)
print("Columnas usadas como features:", columnas_usadas)

filas_completas = features.notna().all(axis=1) & target.notna()
features_limpio = features.loc[filas_completas].copy()
target_limpio = target.loc[filas_completas].copy()

print(f"\nRegistros tras eliminar NaN: {len(features_limpio):,}")
print(f"  Entregados a train/test: X = {len(features_limpio):,} filas, {len(columnas_usadas)} columnas")

Columnas usadas como features: ['VIC_SEXO', 'VIC_EDAD', 'VIC_ALFAB', 'VIC_ESCOLARIDAD', 'VIC_EST_CIV', 'VIC_GRUPET', 'Departamento', 'ANIO']



Registros tras eliminar NaN: 385,519
  Entregados a train/test: X = 385,519 filas, 8 columnas


## 4. División entrenamiento / prueba (80 % / 20 %)

Se usa **estratificación** por la variable respuesta para mantener la proporción de clases (1/0) en train y test. Semilla fija para reproducibilidad.

In [12]:
PROPORCION_PRUEBA = 0.2
SEMILLA = 42

features_entrenamiento, features_prueba, target_entrenamiento, target_prueba = train_test_split(
    features_limpio,
    target_limpio,
    test_size=PROPORCION_PRUEBA,
    stratify=target_limpio,
    random_state=SEMILLA
)

total = len(features_limpio)
n_train = len(features_entrenamiento)
n_test = len(features_prueba)
print("División entrenamiento / prueba (estratificada)")
print("=" * 60)
print(f"  Entrenamiento: {n_train:,} registros ({n_train/total*100:.1f} %)")
print(f"  Prueba:       {n_test:,} registros ({n_test/total*100:.1f} %)")
print(f"  Total:        {total:,} registros")

División entrenamiento / prueba (estratificada)
  Entrenamiento: 308,415 registros (80.0 %)
  Prueba:       77,104 registros (20.0 %)
  Total:        385,519 registros


## 5. Análisis de balance de la variable respuesta

Proporción de cada clase (1 = violencia física, 0 = no) en el dataset global, en entrenamiento y en prueba. La estratificación mantiene proporciones similares en ambos conjuntos.

In [13]:
ETIQUETAS_CLASE = {1: "1 (violencia física)", 0: "0 (no violencia física)"}


def reportar_balance(target_global, target_train, target_test):
    def resumen(serie, etiqueta):
        conteo = serie.value_counts().sort_index()
        total = len(serie)
        print(f"\n  {etiqueta} (n = {total:,})")
        for clase in conteo.index:
            etiq = ETIQUETAS_CLASE.get(clase, clase)
            pct = conteo[clase] / total * 100
            print(f"    {etiq}: {conteo[clase]:,} ({pct:.1f} %)")

    print("Balance de la variable VIOLENCIA_FISICA (1 = violencia física, 0 = no)")
    print("=" * 60)
    resumen(target_global, "Dataset completo (antes del split)")
    resumen(target_train, "Entrenamiento")
    resumen(target_test, "Prueba")

    pct_clase_1 = (target_global == 1).mean() * 100
    if 40 <= pct_clase_1 <= 60:
        balance = "balanceado (proporciones cercanas a 50/50)"
    else:
        balance = "desbalanceado (una clase domina)"
    print(f"\nConclusión: el dataset está {balance}.")


reportar_balance(target_limpio, target_entrenamiento, target_prueba)

Balance de la variable VIOLENCIA_FISICA (1 = violencia física, 0 = no)

  Dataset completo (antes del split) (n = 385,519)
    0 (no violencia física): 153,789 (39.9 %)
    1 (violencia física): 231,730 (60.1 %)

  Entrenamiento (n = 308,415)
    0 (no violencia física): 123,031 (39.9 %)
    1 (violencia física): 185,384 (60.1 %)

  Prueba (n = 77,104)
    0 (no violencia física): 30,758 (39.9 %)
    1 (violencia física): 46,346 (60.1 %)

Conclusión: el dataset está desbalanceado (una clase domina).


In [14]:
tabla_balance = pd.DataFrame({
    'Conjunto': ['Completo', 'Entrenamiento', 'Prueba'],
    'N': [len(target_limpio), len(target_entrenamiento), len(target_prueba)],
    'Clase 1 (n)': [
        (target_limpio == 1).sum(),
        (target_entrenamiento == 1).sum(),
        (target_prueba == 1).sum()
    ],
    'Clase 0 (n)': [
        (target_limpio == 0).sum(),
        (target_entrenamiento == 0).sum(),
        (target_prueba == 0).sum()
    ]
})
tabla_balance['% clase 1'] = (tabla_balance['Clase 1 (n)'] / tabla_balance['N'] * 100).round(1)
tabla_balance['% clase 0'] = (tabla_balance['Clase 0 (n)'] / tabla_balance['N'] * 100).round(1)
print("Resumen para el informe (1 = violencia física, 0 = no):")
display(tabla_balance)

Resumen para el informe (1 = violencia física, 0 = no):


,Conjunto,N,Clase 1 (n),Clase 0 (n),% clase 1,% clase 0
0,Completo,385519,231730,153789,60.1,39.9
1,Entrenamiento,308415,185384,123031,60.1,39.9
2,Prueba,77104,46346,30758,60.1,39.9


### Resumen para el informe (Punto 5)

Se realizó una división aleatoria estratificada del dataset utilizando la variable respuesta VIOLENCIA_FISICA, con el objetivo de mantener la proporción de las clases en los conjuntos de entrenamiento y prueba.

El dataset final utilizado para el modelado contiene 385,519 registros, luego de eliminar filas con valores faltantes en las variables seleccionadas.

La división se realizó utilizando una proporción de:

80 % de los datos para entrenamiento

20 % de los datos para prueba

Para garantizar la reproducibilidad del experimento se utilizó una semilla fija (random_state = 42).

La estratificación permitió mantener proporciones similares de las clases “Sí” y “No” en ambos conjuntos.

En el dataset completo la distribución de clases es aproximadamente:

Sí (violencia física): 60.1 %

No (sin violencia física): 39.9 %

Debido a que ninguna de las clases supera una proporción extremadamente dominante, el dataset presenta un leve desbalance, pero no lo suficientemente alto como para requerir técnicas adicionales de balanceo en esta etapa del análisis.